# Brain tumor MRI classification — reproducible capstone experiments
**Disci, Gurcan & Soylu (2025), Cancers 17(1), 121**  
**Team:** Vishesh Rao, Pranay Vishwakarma, Manu Vahan · **M2/M3:** 6 October 2026

This notebook uses the shared repository pipeline and displays **actual completed runs**. Published reference values, smoke checks, and measured results are kept separate. It defaults to analysis of saved results, so the presentation does not start a long training job.

### Introduction and hypothesis
Transfer learning offers a practical four-class MRI classifier under limited compute. Our literature survey emphasizes class-sensitive performance, computational cost, and evaluation reliability. We reproduce the paper's standard CNN comparison and separately audit known train/test duplicates. Image-level results cannot establish generalization to unseen patients without subject identifiers.

**Operational hypothesis:** Under an identical leakage-audited protocol, MobileNetV2's test macro-F1 is no more than **0.02 below Xception's**, with fewer total model parameters. This is a preliminary single-seed comparison, not a statistical non-inferiority claim.

[Paper](https://doi.org/10.3390/cancers17010121) · [Exact dataset version 1](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset/versions/1)

### Local / free Colab setup
Locally, open this notebook from the repository using its Python 3.11/3.12 environment. For Colab, select **T4 GPU** and runtime version **2026.07 (Python 3.12)** in Runtime → Change runtime type, then upload **brain_mri_colab.zip** to `/content` using the Files sidebar. The next cell extracts our code bundle and installs the pinned dependencies only in Colab. MRI images and heavy checkpoints are downloaded/generated separately. If dependencies were already imported in Colab before installation, restart the runtime before continuing.

**Scope:** replicate the selected Xception baseline first, then test MobileNetV2 as a smaller replacement. The four other paper architectures are outside this experiment. Colab evidence contains the baseline, the matched audited pair, and one supplementary paper-based MobileNetV2 run.

In [ ]:
import json
import os
import sys
from pathlib import Path

IN_COLAB = Path("/content").exists() and "google.colab" in sys.modules
if IN_COLAB:
    if sys.version_info >= (3, 13):
        raise RuntimeError(
            "Select Colab runtime version 2026.07 (Python 3.12) and T4 GPU in Runtime > Change runtime type, then reconnect and rerun."
        )
    import subprocess
    from zipfile import ZipFile

    bundle = Path("/content/brain_mri_colab.zip")
    root = Path("/content/brain_mri")
    if not (root / "configs/disci2025.yaml").exists():
        if not bundle.exists():
            raise FileNotFoundError("Upload brain_mri_colab.zip to the Colab Files sidebar first.")
        with ZipFile(bundle) as archive:
            for member in archive.infolist():
                if not (root / member.filename).resolve().is_relative_to(root.resolve()):
                    raise ValueError("Unsafe bundle path")
            archive.extractall(root)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(root / "requirements.txt")]
    )
    ROOT = root
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    ROOT = next((p for p in candidates if (p / "configs/disci2025.yaml").exists()), None)
    if ROOT is None:
        raise FileNotFoundError("Open the notebook from the extracted project/repository folder.")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ.setdefault("KERAS_HOME", str(ROOT / ".cache/keras"))
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache/matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(ROOT / ".cache"))
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
print("Project:", ROOT)
print("Mode:", "Colab" if IN_COLAB else "local")

## 1. Settings and experiment identity
The paper reports **128 × 128 × 3**, Adam **0.0001**, batch **20**, **five epochs**, and a common Flatten → Dropout(0.3) → Dense(128, ReLU) → Dropout(0.2) → Softmax(4) head. Seed 42, Gaussian kernel 5, threshold 45, bounding-rectangle crop, and full backbone fine-tuning are disclosed assumptions.

`paper_based` preserves the official split and reports clean and seeded transformed test views. `leakage_audited` removes exact overlap and redundant training images, uses 15% validation, and augments training only. Perceptual candidates remain unresolved; no patient independence is asserted.

The optional all-model switch adds the other four paper backbones. Changing normalization, trainability, sample counts, or epochs is a **different experiment**, not an exact reproduction.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

from scripts.reproduce_disci2025 import download_dataset, prepare, refresh_results
from src.reproduction.artifacts import paired_experiment_identity
from src.reproduction.config import CLASSES, environment_metadata, load_config
from src.reproduction.evaluation import PAPER, classification_metrics
from src.reproduction.training import configure_runtime

CONFIG = load_config(ROOT / "configs/disci2025.yaml")
CONFIG["data_dir"] = str(ROOT / "data/raw")
CONFIG["output_dir"] = str(ROOT / "outputs/disci2025")
RUN_TRAINING = True
MODELS = ["Xception", "MobileNetV2"]

if RUN_TRAINING:
    analysis_environment = configure_runtime(CONFIG["seed"], ROOT / ".cache")
else:
    analysis_environment = {
        **environment_metadata(),
        "devices": ["Analysis only; training hardware is recorded in provenance."],
    }
display(pd.Series(CONFIG, name="experiment settings"))
print("Notebook analysis devices:", analysis_environment["devices"])
print("Training hardware and versions are recorded separately in each run provenance.")
assert any("GPU" in device for device in analysis_environment["devices"]), (
    "Select T4 GPU before full training."
)

## 2. Exact dataset and leakage audit
**Pin version 1.** The current version 2 has 7,200 images and does not match the paper's 7,023. All eight class/partition counts are checked before training. The archive checksum and public source are included in the deliverable.

Exact decoded-pixel hashes identify duplicated content independently of filenames. Perceptual hashes only propose pairs for review; they do not identify patients.

In [ ]:
source = json.loads((ROOT / "results/disci2025/dataset_source.json").read_text())
audit = json.loads((ROOT / "results/disci2025/data_audit.json").read_text())
counts = pd.DataFrame(
    {"class": CLASSES, "Training": [1321, 1339, 1595, 1457], "Testing": [300, 306, 405, 300]}
)
display(counts)
print("Dataset:", source["dataset"], "version", source["version"])
print("Archive SHA256:", source["archive_sha256"])
display(
    pd.Series(
        {
            k: audit[k]
            for k in [
                "cross_split_exact_count",
                "within_train_exact_redundant_count",
                "within_test_exact_redundant_count",
                "patient_ids_available",
            ]
        },
        name="audit",
    )
)
print("Unreviewed perceptual candidate pairs:", len(audit["near_duplicate_candidates"]))
print("Known cross-split duplicate images are excluded only in the separate audited protocol.")

## 3. Preprocessing example
Largest-contour cropping extracts foreground, which may be the brain rather than a tumor. A missing contour falls back to the full slice. No lesion segmentation claim is made. Raw examples are displayed only when MRI files are available; the Colab bundle keeps the measured-result evidence but excludes the large dataset.

In [ ]:
import cv2

from src.reproduction.data import inventory_dataset
from src.reproduction.preprocessing import preprocess_image

if (Path(CONFIG["data_dir"]) / "Training").exists():
    inventory = inventory_dataset(CONFIG["data_dir"])
    examples = inventory[inventory.original_partition == "train"].groupby("label").head(1)
    fig, axes = plt.subplots(4, 2, figsize=(7, 10))
    for row, item in enumerate(examples.itertuples()):
        path = Path(CONFIG["data_dir"]) / item.path
        original = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        processed, info = preprocess_image(path, CONFIG)
        axes[row, 0].imshow(original, cmap="gray")
        axes[row, 0].set_title(item.class_name + " · original")
        axes[row, 1].imshow(processed)
        axes[row, 1].set_title("Foreground crop · 128 × 128")
        for axis in axes[row]:
            axis.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print(
        "Raw images are not bundled. Set RUN_TRAINING=True to download them before new experiments."
    )

## 4. Optional training — baseline before proposed changes
Keep `RUN_TRAINING=False` for presentation: the completed Colab evidence is already saved. New training first replicates paper-based Xception, then trains audited Xception and MobileNetV2 with identical data/settings. Audited runs start independently from ImageNet weights. `resume=True` skips verified completed checkpoints; interrupted runs restart from epoch 1. The other four paper models are outside scope.

In [ ]:
if RUN_TRAINING:
    from src.reproduction.training import train_model

    if not (Path(CONFIG["data_dir"]) / "Training").exists():
        download_dataset(Path(CONFIG["data_dir"]))
    experiments = [
        ("paper_based", "Xception"),
        ("leakage_audited", "Xception"),
        ("leakage_audited", "MobileNetV2"),
    ]
    for protocol, model_name in experiments:
        config = {**CONFIG, "protocol": protocol}
        manifests = prepare(config)
        run_dir = Path(config["output_dir"]) / protocol / model_name
        train_model(model_name, manifests, config, run_dir, resume=True)
        refresh_results(Path(config["output_dir"]))
else:
    print("Training disabled: analyzing saved results only. Smoke runs are excluded.")

## 5. Published targets and measured results
**Important correction:** Xception's published testing accuracy is **95.27%**. The headline **98.73%** is consistent with a sample-count-weighted combination of training and test accuracy. It is not a held-out metric. The formula is inferred from the tables; source inconsistencies are retained rather than forced into agreement.

The reference table below contains source values. The next table contains actual completed MRI runs. Differences are in percentage points and retain each protocol and test view.

In [ ]:
display(PAPER[PAPER.model.isin(MODELS)].round(4))
live_outputs = Path(CONFIG["output_dir"])
compact_outputs = ROOT / "results/disci2025/runs"
if RUN_TRAINING or not compact_outputs.exists():
    refresh_results(live_outputs)
EVIDENCE_DIR = (
    live_outputs if RUN_TRAINING else compact_outputs if compact_outputs.exists() else live_outputs
)
refresh_results(EVIDENCE_DIR)
summary_path = ROOT / "results/disci2025/summary.csv"
summary = (
    pd.read_csv(summary_path)
    if summary_path.exists() and summary_path.stat().st_size > 2
    else pd.DataFrame()
)
if summary.empty:
    print(
        "No accepted full experiment results yet. This notebook does not fabricate reproduced scores."
    )
else:
    columns = [
        "model",
        "protocol",
        "evaluation_view",
        "n_train",
        "n_test",
        "epochs",
        "accuracy",
        "macro_f1",
        "weighted_f1",
        "balanced_accuracy",
        "macro_roc_auc",
        "test_accuracy_difference_pp",
    ]
    display(summary[columns].round(4))
    completed = set(summary.loc[summary.protocol == "paper_based", "model"])
    display(
        pd.DataFrame(
            {
                "paper model": CONFIG["models"],
                "full paper-based run completed": [m in completed for m in CONFIG["models"]],
            }
        )
    )

## 6. Learning curves and confusion matrices
These figures are generated from saved histories and clean held-out predictions, not copied from the paper. Learning curves show training and diagnostic validation only; test accuracy is not used to select a model.

In [ ]:
if not summary.empty:
    for row in summary[summary.evaluation_view == "clean"].itertuples():
        folder = EVIDENCE_DIR / row.run_dir
        print(row.protocol, "·", row.model)
        for filename in ["learning_curves.png", "confusion_matrix.png"]:
            if (folder / filename).exists():
                display(Image(filename=str(folder / filename), width=700))

## 7. Verify metrics from saved predictions
We recompute macro-F1 and accuracy from exported probabilities, and check all 1,311 official test images are represented. This protects against misordered labels and hand-entered result tables. The duplicate test images remain disclosed.

In [ ]:
if not summary.empty:
    verification = []
    for row in summary.itertuples():
        path = EVIDENCE_DIR / row.run_dir / f"predictions_{row.evaluation_view}.csv"
        saved = pd.read_csv(path)
        probabilities = saved[[f"p_{name}" for name in CLASSES]].to_numpy()
        metrics = classification_metrics(saved.label.to_numpy(), probabilities)
        assert len(saved) == row.n_test == 1311
        assert np.isclose(metrics["accuracy"], row.accuracy)
        assert np.isclose(metrics["macro_f1"], row.macro_f1)
        verification.append(
            {
                "model": row.model,
                "protocol": row.protocol,
                "view": row.evaluation_view,
                "prediction rows": len(saved),
                "recomputed metrics match": True,
            }
        )
    display(pd.DataFrame(verification))

## 8. Resource comparison and hypothesis outcome
Counts include the head. Training time excludes data/weight downloads and final evaluation. Inference timing, when available, uses warmed-up batch-1 execution on the same device and excludes preprocessing. Single-seed outcomes are preliminary.

In [ ]:
if not summary.empty:
    resource_columns = ["model", "protocol", "total_parameters", "training_seconds"]
    for column in ["median_inference_ms", "p95_inference_ms"]:
        if column in summary:
            resource_columns.append(column)
    display(summary[summary.evaluation_view == "clean"][resource_columns].round(3))
    pair = summary[
        (summary.protocol == "leakage_audited") & (summary.evaluation_view == "clean")
    ].set_index("model")
    if {"MobileNetV2", "Xception"}.issubset(pair.index):
        mobile, xception = pair.loc["MobileNetV2"], pair.loc["Xception"]
        provenance = json.loads((ROOT / "results/disci2025/provenance.json").read_text())["runs"]
        identities = {r["model"]: r for r in provenance if r["protocol"] == "leakage_audited"}
        assert paired_experiment_identity(identities["MobileNetV2"], identities["Xception"]), (
            "Different data or shared settings: no paired conclusion"
        )
        assert mobile.n_train == xception.n_train
        assert mobile.epochs == xception.epochs == 5
        assert mobile.seed == xception.seed
        gap = float(xception.macro_f1 - mobile.macro_f1)
        supported = gap <= 0.02 and mobile.total_parameters < xception.total_parameters
        display(
            pd.Series(
                {
                    "Xception minus MobileNetV2 macro-F1": gap,
                    "allowed margin": 0.02,
                    "MobileNetV2 uses fewer total parameters": bool(
                        mobile.total_parameters < xception.total_parameters
                    ),
                    "hypothesis supported in this single-seed experiment": bool(supported),
                }
            )
        )
    else:
        print("Hypothesis outcome pending: both full audited model runs are required.")

## 9. Conclusions and presentation boundaries
- Explain the correct dataset **version**, the actual completed runs, and ordinary held-out metrics.
- Explain the **263 exact overlapping training images** and how the audited protocol removes them.
- Interpret our lightweight-model hypothesis using the actual paired results; report an unfavorable result if obtained.
- Keep the paper's combined train/test statistic separate from accuracy on unseen images.
- Do not claim exact numerical replication, patient-independent generalization, validated tumor localization, clinical validity, or completion of models marked pending.

**Limitations:** underspecified paper preprocessing/trainability; GPU/library differences; one seed; missing patient IDs; unresolved perceptual candidates; duplicate test images; several changes between original and audited protocols. Future work includes repeated seeds, grouped/external validation, class-weighting or normalization ablations, and annotation-based explanations.

**Report:** `docs/report/Disci2025_Reproduction_Addendum.md` contains the introduction, hypothesis, methods, audit, measured results and limitations. The original literature-survey PDF is preserved.

**Measured conclusion:** Xception baseline clean accuracy is **93.67%** (paper: **95.27%**); transformed accuracy is **93.29%**. Audited macro-F1 is **0.9176 for Xception** versus **0.7372 for MobileNetV2**. The **0.1804** gap exceeds the **0.02** margin, so the hypothesis is **not supported** in this single-seed experiment. MobileNetV2 uses **80.52% fewer parameters**. An unfavorable hypothesis result is valid evidence and should be presented as such.

In [ ]:
if IN_COLAB and RUN_TRAINING:
    import zipfile

    from google.colab import files

    archive = Path("/content/disci2025_results.zip")
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as target:
        for directory in ["results/disci2025", "outputs/disci2025"]:
            for path in (ROOT / directory).rglob("*"):
                if path.is_file() and path.suffix in {".json", ".csv", ".png"}:
                    target.write(path, path.relative_to(ROOT))
    files.download(str(archive))